### 自定义输出解析
---
1. 纳美星语解析器
2. 布尔值解析器

#### 1. 纳美星语解析器
---

In [ ]:
# 注意：如果没有 langchain_deepseek，可以使用 langchain_openai.ChatOpenAI 替代（见下方说明）
try:
    from langchain_deepseek import ChatDeepSeek
except ImportError:
    from langchain_openai import ChatOpenAI as ChatDeepSeek # 兼容回退方案

from langchain_core.messages import AIMessage
import re
import os
from dotenv import load_dotenv

load_dotenv()

# 1. 初始化模型
model = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # DeepSeek 官方模型名通常为 deepseek-chat 或 deepseek-reasoner
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"), 
    temperature=0,
)

def parse(ai_message: AIMessage) -> str:
    """Convert message to Na'vi style language."""
    content = ai_message.content
    if not content:
        return "Oel ngati kameie... (No content)"

    # 纳美语特征转换规则
    navi_rules = {
        # 常见问候语转换
        r'\b[Hh]ello\b': 'Kaltxi',
        r'\b[Hh]i\b': 'Kaltxi',
        r'\b[Tt]hank you\b': 'Irayo',
        r'\b[Gg]oodbye\b': 'Eywa ngahu',

        # 添加纳美语特征的后缀/替换
        r'\b(friend|friends)\b': 'eylan',
        r'\b(brother|brothers)\b': 'tsmukan',
        r'\b(sister|sisters)\b': 'tsmuke',

        # 添加纳美语语气词 (匹配句末标点)
        r'([!.])\s*$': r'\1 kxe!',
        r'(\?)\s*$': r'\1 srak?',

        # 在某些词前添加特征前缀
        r'\b(beautiful|pretty)\b': 'na\'vi',
        r'\b(sacred|holy)\b': 'txe\'lan',
    }

    # 应用转换规则
    result = content
    for pattern, replacement in navi_rules.items():
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)

    # 添加纳美语特征词缀 (Oel ngati kameie = I see you)
    result = f"Oel ngati kameie... {result}"
    return result

# 2. 使用 LCEL 语法组合成一个简单的链
chain = model | parse

# 3. 调用链
try:
    output = chain.invoke("Hello my friend, how are you?")
    print("✅ 纳美语转换结果:")
    print(output)
except Exception as e:
    print(f"❌ 调用失败: {e}")

✅ 纳美语转换结果:
Oel ngati kameie... Kaltxi! I’m doing well, thanks for asking. 😊 How are you? What can I help you with today? srak?


2. 布尔值解析器
---

In [25]:
# 通过集成BaseOutputParser来实现
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import BaseOutputParser

# [bool] 描述了一个泛型的参数化。
# 它基本上说明了解析的返回类型是什么
# 在这种情况下，返回类型为 True 或 False
# 自定义解析器将YES、NO字符串解析为布尔值
class BooleanOutputParser(BaseOutputParser[bool]):
    """Custom boolean parser."""

    true_val: str = "YES"
    false_val: str = "NO"

    def parse(self, text: str) -> bool:
        cleaned_text = text.strip().upper()
        if cleaned_text not in (self.true_val.upper(), self.false_val.upper()):
            raise OutputParserException(
                f"BooleanOutputParser expected output value to either be "
                f"{self.true_val} or {self.false_val} (case-insensitive). "
                f"Received {cleaned_text}."
            )
        return cleaned_text == self.true_val.upper()

    # 设置统一接口名
    @property
    def _type(self) -> str:
        return "boolean_output_parser"
    
# 简单调用
parser = BooleanOutputParser()
parser.invoke("YES")

# 如果传入的值不是YES NO会报错
# try:
#     parser.invoke("MEOW")
# except Exception as e:
#     print(f"Triggered an exception of type: {type(e)}")

# 我们也可以更换参数
parser = BooleanOutputParser(true_val="OKAY", false_val="NOKY")
parser.invoke("NOKY")




False